# Multi-Factor Portfolio Construction

> **Universe & leverage:** **Top 40 market-cap universe**, **200% gross exposure**

This notebook demonstrates how to replicate multi-factor portfolios using the [Aperiodic Factors API](https://factors.aperiodic.io).

Licence: MIT

---

### About Aperiodic Factors
[Aperiodic Factors](https://factors.aperiodic.io) provides proprietary cross-sectional factors and market-neutral, multi-factor portfolios for digital assets.

---

### **Prerequisites to run this notebook**

Create an account at [Aperiodic Factors](https://factors.aperiodic.io) and generate an API key in your API settings. Make it available to the notebook as the `APERIODIC_API_KEY` environment variable (for example via a `.env` file) — see the *Run the notebooks (setup)* section of the repository README.

### Import Libraries

In [ ]:
# The Aperiodic Factors client is vendored in this repository
# (aperiodic-factors-client/) rather than published to PyPI, so we install it
# from its local path. `pip install -r requirements.txt` (see the README) already
# does this for you; this cell makes the notebook self-contained if you haven't.
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "aperiodic-factors-client").is_dir() and _root != _root.parent:
    _root = _root.parent

!{sys.executable} -m pip install alphalens-reloaded "{_root}/aperiodic-factors-client"

In [ ]:
from __future__ import annotations

import os
from collections.abc import Generator
from dataclasses import dataclass
from datetime import UTC, datetime
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from joblib import Parallel, delayed
import aperiodic

### Main Specification

Modify the parameters below to replicate the portfolio.
See the full list of available [factors and portfolios](https://factors.aperiodic.io/portfolios) in the catalog.

- **APERIODIC_API_KEY**: Create an account at [Aperiodic Factors](https://factors.aperiodic.io) and generate an API key in your API settings. The notebook reads it from the `APERIODIC_API_KEY` environment variable (e.g. a `.env` file), or from a Google Colab secret of the same name.
- **factors**: Factors to include in the multi-factor portfolio.
- **start_date**: Start date for the backtest.
- **end_date**: End date for the backtest.
- **universe_size**: The number of tickers to include in the portfolio.
- **allocation_cap**: The maximum allocation for any single ticker in the portfolio.

In [ ]:

try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata

    APERIODIC_API_KEY = userdata.get("APERIODIC_API_KEY")
else:
    load_dotenv()
    APERIODIC_API_KEY = os.getenv("APERIODIC_API_KEY")

factors = [
    "retail_flow",
    "polaris",
    "instantaneous_momentum",
    "altair",
    "carry_enhanced",
    "margin_risk",
]
start_date = "2020-01-01"
end_date = "2025-08-01"
universe_size = "40"
allocation_cap = 0.2

print(
    "**Parameters:**\n"
    f"factors: {factors}\n"
    f"start_date: {start_date}\n"
    f"end_date: {end_date}\n"
    f"universe_size: {universe_size}\n"
    f"allocation_cap: {allocation_cap}\n"
)

### Helper Functions for Portfolio Construction

Defining inverse volatility weighting method and helper functions.

In [ ]:

def calculate_inverse_volatility_weighting(
    underlying: pd.DataFrame, weights: pd.DataFrame, period: int
) -> pd.DataFrame:
    stds = underlying.rolling(period, min_periods=0).std()
    stds = stds.where(weights.notna())

    std_inverse = 1 / stds.div(stds.sum(axis="columns"), axis="index")
    std_inverse[std_inverse == np.inf] = 0.0
    return std_inverse.div(std_inverse.sum(axis="columns"), axis="index").mul(
        weights.count(axis="columns"), axis="index"
    )

def filter_none(results: list[Any]) -> list[Any]:
    return [r for r in results if r is not None]

### Backtest Functions

Core portfolio backtesting and performance calculation.

In [ ]:

ReturnsDataFrame = pd.DataFrame
Returns = pd.Series
Signal = pd.Series


@dataclass
class PortfolioBacktestResult:
    portfolio_returns: Returns
    component_returns: ReturnsDataFrame
    transaction_costs: pd.Series
    lag: int

    def split(self, start_date, end_date) -> PortfolioBacktestResult:
        return PortfolioBacktestResult(
            portfolio_returns=self.portfolio_returns[start_date:end_date],
            component_returns=self.component_returns[start_date:end_date],
            transaction_costs=self.transaction_costs[start_date:end_date],
            lag=self.lag,
        )


def backtest_portfolio(
    weights: pd.DataFrame,
    underlying: ReturnsDataFrame,
    transaction_cost: float,
    lag: int,
) -> PortfolioBacktestResult:
    """
    Create returns from a signal and a target.
    """
    assert weights.columns.equals(underlying.columns), "Columns must match"
    underlying = underlying.loc[weights.index]
    weights = weights.fillna(0.0).reindex(underlying.index).ffill().copy()
    weights.columns = underlying.columns
    delta_pos = weights.diff(1).abs().fillna(0.0)
    costs = transaction_cost * delta_pos
    returns = (underlying * weights.shift(lag)) - costs
    portfolio_returns = returns.sum(axis="columns")

    transaction_costs = costs.sum(axis="columns")
    return PortfolioBacktestResult(
        portfolio_returns=portfolio_returns,
        component_returns=returns,
        transaction_costs=transaction_costs,
        lag=lag,
    )


def create_cross_sectional_bins(
    factor_data: pd.DataFrame, quantiles: int
) -> tuple[pd.DataFrame, pd.DataFrame]:
    factors_binned = factor_data.apply(
        lambda x: pd.qcut(x, q=quantiles, labels=False, duplicates="drop"),
        axis="columns",
    )
    weights = factors_binned.div(factors_binned.max(axis="columns"), axis="index")
    weights = weights.mul(2).sub(1)

    return weights.div(weights.abs().sum(axis="columns"), axis="index"), factors_binned


def average_elementwise(
    dfs: list[pd.DataFrame],
) -> pd.DataFrame:
    return pd.concat(dfs).groupby(level=0).mean().sort_index()


def plot_last_row_barplot(df: pd.DataFrame, title: str):
    last_row = df.iloc[-1].dropna().sort_values()
    plt.bar(last_row.index, last_row.values)
    plt.title(title)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()


def to_drawdown(prices: pd.Series) -> pd.Series:
    running_max = prices.cummax()
    return (prices - running_max) / running_max


def rebase(prices: pd.Series) -> pd.Series:
    """Rebase a price series to 1.0"""
    return prices / prices.iloc[0]


def plot_backtest_results(
    cumulative_returns: pd.Series,
    benchmark: pd.Series | None,
    portfolio: str,
    figsize=(12, 10),
):
    """
    Plot backtest results with performance chart and signal.

    Args:
        cumulative_returns (pd.Series): Series containing cumulative returns
        benchmark (pd.Series): Series containing benchmark data
        portfolio (str): Portfolio name/identifier
        figsize (tuple): Figure size as (width, height) in inches
    """
    show_benchmark = benchmark is not None
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=figsize, gridspec_kw={"height_ratios": [2, 1]}
    )

    ax1.plot(
        cumulative_returns.index,
        cumulative_returns,
        label="Portolio Cumulative Returns",
        color="darkBlue",
    )
    if show_benchmark:
        ax1.plot(
            benchmark.index,
            benchmark,
            label=f"Benchmark ({benchmark.name})",
            color="gray",
        )
    ax1.set_title("Performance of Portfolio", fontsize=14)
    ax1.legend()
    ax1.grid(True, axis="y", linestyle="--")

    ax2.plot(
        cumulative_returns.index,
        to_drawdown(cumulative_returns),
        label="Drawdown Portfolio",
        color="red",
    )
    if show_benchmark:
        ax2.plot(
            benchmark.index,
            to_drawdown(benchmark),
            label="Drawdown Benchmark",
            color="gray",
        )
    ax2.set_title(f"Portfolio {portfolio}")
    ax2.legend()
    ax2.grid(True, axis="y", linestyle="--")

    plt.tight_layout()
    plt.show()

    return fig, (ax1, ax2)

### Retrieving the historical universe (Top 40 Market Cap Crypto Assets)

To avoid any lookahead bias, the universe needs to be constructed on a rolling basis. We apply heavy volume & volatility filters to reduce the turnover of the universe further.

In [ ]:
universe = aperiodic.get_historical_universe(
    size=universe_size,
    start_date=start_date,
    end_date=end_date,
    api_key=APERIODIC_API_KEY,
)
tickers: list[Any] = list(universe.columns)

### Retrieve Price Data

Fetching underlying asset price data for backtesting.

In [ ]:
underlying = aperiodic.get_prices(
    tickers=tickers,
    api_key=APERIODIC_API_KEY,
    start_date=start_date,
    end_date=end_date,
)

# Filter out the tickers that don't have historical data on Binance
tickers = pd.Index(tickers).intersection(underlying.columns).to_list()

print(f"Number of tickers: {len(tickers)}")
print(f"Tickers: {tickers}")

### Fetching historical (raw) factor data

The portfolio combines 6 proprietary cross-sectional factors:
- [Retail Flow](https://factors.aperiodic.io/portfolio/retail_flow.40?exchange=unconstrained)
- [Polaris](https://factors.aperiodic.io/portfolio/polaris.40?exchange=unconstrained)
- [Instantaneous Momentum](https://factors.aperiodic.io/portfolio/instantaneous_momentum.40?exchange=unconstrained)
- [Altair](https://factors.aperiodic.io/portfolio/altair.40?exchange=unconstrained)
- [Enhanced Carry](https://factors.aperiodic.io/portfolio/carry_enhanced.40?exchange=unconstrained)
- [Margin Risk](https://factors.aperiodic.io/portfolio/margin_risk.40?exchange=unconstrained)

In [ ]:
historical_factors = [
    aperiodic.get_portfolio_factors_historical(
        id=factor,
        tickers=tickers,
        api_key=APERIODIC_API_KEY,
        start_date=start_date,
        end_date=end_date,
    ).where(universe)
    for factor in factors
]

In [ ]:
# Using `alphalens` to display a comprehensive analysis of the first factor (retail_flow)
import warnings

import alphalens
import pandas as pd

# `alphalens` expects numeric inputs; object dtypes from the API can later break
# pandas reductions inside the tear sheet with `ufunc sqrt` errors.
factor_series = pd.to_numeric(historical_factors[0].stack(), errors="coerce").dropna()
price_frame = underlying.apply(pd.to_numeric, errors="coerce").sort_index().sort_index(axis=1)

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="The default fill_method='pad' in DataFrame.pct_change is deprecated",
        category=FutureWarning,
    )
    factor_data = alphalens.utils.get_clean_factor_and_forward_returns(
        factor_series.sort_index(),
        price_frame,
        quantiles=5,
        max_loss=0.5,
    )
    alphalens.tears.create_full_tear_sheet(factor_data)

### Factor Processing

The "factors" here are the raw data - they need to be translated into portfolio weights.

We use quantile transformation to get the factors to the same uniform distribution (there should be equal number of assets in each bin) - and then use simple arithmetic averaging to ensemble them.

The idea here is to long / short portfolio weights in proportion to the factor values.

In [ ]:
binned_factors = [
    create_cross_sectional_bins(historical_factor, quantiles=20)[0]
    for historical_factor in historical_factors
]

is_single_factor = len(factors) == 1
if is_single_factor:
    factor_ensemble = binned_factors[0]
    plot_last_row_barplot(factor_ensemble, f"Single Factor: {factors[0]}")
else:
    # Simple arithmetic average of the different factors
    factor_ensemble = average_elementwise(binned_factors)
    plot_last_row_barplot(factor_ensemble, "Average of Factors")

To center / balance the multi-factor portfolio, we apply a final cross-sectional binning transformation.

In [ ]:
if is_single_factor:
    binned_portfolio = factor_ensemble
else:
    binned_portfolio = create_cross_sectional_bins(factor_ensemble, quantiles=20)[0]

    plot_last_row_barplot(binned_portfolio, "Final binning phase")

### Portfolio Construction

By using inverse volatility weighting, we can get to a portfolio that is more aware of the nuances between the volatility of different assets. A memecoin's volatility is widely different to Bitcoin's, and the former should have a smaller allocation if we want to "equalize" their risk contribution.

In [ ]:
risk_weights = calculate_inverse_volatility_weighting(
    underlying.pct_change(), binned_portfolio, period=120
)
portfolio_weights = binned_portfolio.mul(risk_weights, axis="columns")
portfolio_weights = portfolio_weights.clip(lower=-allocation_cap, upper=allocation_cap)

plot_last_row_barplot(portfolio_weights, "Inverse Volatility Weighting")

### Portfolio Backtesting

Running the vectorized backtest with reasonable transaction costs (5bps). This ignores slippage & funding costs - the goal of the function is to be easily auditable.
The assumption is that if one wants to trade it, they'll want to use their own or trusted (more sophisticated) backtesting engine, that we can't easily replicate here.

In [ ]:
result = backtest_portfolio(
    weights=portfolio_weights,
    underlying=underlying[portfolio_weights.columns].pct_change(),
    transaction_cost=0.0005,
    lag=1,
)

### Results Visualization

Plotting the cumulative portfolio returns.

In [ ]:
plot_backtest_results(
    rebase((1 + result.portfolio_returns).cumprod()),
    None,
    "-".join(factors),
)

In [ ]:

sharpe_ratio = (
    result.portfolio_returns.mean() / result.portfolio_returns.std() * np.sqrt(365)
)
daily_turnover = portfolio_weights.fillna(0.0).diff(1).abs().sum(axis="columns").mean()
print(f"Sharpe ratio: {sharpe_ratio}")
print(f"Daily turnover: {daily_turnover.mean()*100:.2f}%")

The goal of this notebook is to demonstrate that it may not take much to actually create a multi-factor portfolio with great risk-adjusted returns in crypto, purposefully using the simplest possible techniques.

A couple of ideas to try out:
- Some of the factors may have too high turnover by default. Smoothing them can be beneficial
- The factors come as "raw data", without any model involved - some have assymetric expected returns in the top vs bottom quantiles. Training a simple model in a walk-forward manner may help recognize this in time, before the ensembling step.
- Experimenting with different, more sophisticated factor ensembling techniques
- Applying more sophisticated portfolio construction techniques
- Portfolio level volatility targeting
- Applying a trend filter on the multi-factor portfolio returns (or on individual factors)
- Does the portfolio still work if the execution is delayed by a day? How does the alpha decay?
- Try various techniques to reduce turnover (and therefore, transaction costs)

Create an account at [Aperiodic Factors](https://factors.aperiodic.io) and generate an API key to access the historical data — so you can run the notebook and start experimenting!